In [ ]:
import torch
import timm

model = torch.load("C:/Users/araya/Downloads/MViTv2_T_in1k.pyth")
print(model)

In [ ]:
# โหลด MViTv2 ด้วย pre-trained weights
model = timm.create_model('mvitv2_base', pretrained=True)
model.eval()  # เปลี่ยนเป็นโหมดประเมินผล

### ----------------------------------------------------------------------------------

### Extract Frames

In [ ]:
import cv2
import os

def extract_frames(video_path, output_dir, frame_rate=5):
    cap = cv2.VideoCapture(video_path)
    count = 0
    success, frame = cap.read()
    while success:
        if count % frame_rate == 0:
            os.makedirs(output_dir, exist_ok=True)
            cv2.imwrite(os.path.join(output_dir, f"frame_{count}.jpg"), frame)
        success, frame = cap.read()
        count += 1
    cap.release()

In [ ]:
extract_frames(video_path="C:/Users/araya/Desktop/keypoints/video_extract/กิน.mp4", output_dir="C:/Users/araya/Desktop/MViT_frame")

In [ ]:
import torchvision

print(torchvision.__version__)

### ------------------------------------------------------

In [26]:
import torch 
import torchvision.transforms as transforms 
from torchvision.utils import make_grid
from PIL import Image
import numpy as np 
import matplotlib.pyplot as plt

In [27]:
def visualize_generator(img_file, transform):
    # Load the requested image
    img = Image.open(img_file)

    # Apply transformations to the image
    imgs = [transform(img) for _ in range(1)]
    imgs = torch.stack(imgs)

    # Display the augmented images in a grid format 
    grid = make_grid(imgs, nrow=2)

    # Convert to numpy for plotting
    grid_np = grid.numpy().transpose(1,2,0)
    grid_np = (grid_np * 255).astype(np.uint8)

    plt.figure(figsize=(4,4))
    plt.axis("off")
    plt.imshow(grid_np)
    plt.show()

In [ ]:
transform = transforms.Compose([
    # transforms.RandomHorizontalFlip(p=0.5),
    # transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=(-10,10)),
    transforms.ToTensor()
])

visualize_generator("C:/Users/araya/Desktop/vscode/pretrained/dataset/train/กลัว/frame19.jpg", transform)

In [ ]:
import os 
import random

test_path = "C:/Users/araya/Desktop/vscode/pretrained/dataset/test"

path = "C:/Users/araya/Desktop/vscode/pretrained/dataset/train"
folder_list = os.listdir(path)
print(folder_list)
for each_folder in folder_list:
    random_angle = random.uniform(-10, 10)
    print(f"Folder: {each_folder}, Angle: {random_angle}")
    # print(each_folder)
    # sub_path = path + "/" + each_folder + ".jpg"
    # print(sub_path)
    file_path = os.path.join(path, each_folder)
    print(file_path)
    frame_list = os.listdir(file_path)
    # print(frame_list)

    transform = transforms.Compose([
    # transforms.RandomHorizontalFlip(p=0.5),
    # transforms.RandomVerticalFlip(p=0.5),
    # transforms.RandomRotation(degrees=(-10,10)),
    transforms.RandomRotation(degrees=(random_angle, random_angle)),
    transforms.ToTensor()
])

    for frame in frame_list:
        # print(frame)
        each_frame_path = path + "/" + each_folder + "/" + frame
        print(each_frame_path)
        visualize_generator(each_frame_path, transform)
        # test_folder = os.mkdir(test_path + "/" + each_folder)

In [ ]:
test_path = "C:/Users/araya/Desktop/vscode/pretrained/dataset/test"
os.mkdir(test_path)

### ---------------------------------------------------------------------------------

### GPT

In [1]:
import torch
import torchvision.transforms as transforms
from torchvision.utils import make_grid
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import os
import random

In [17]:
class AddGaussianNoise(object):
    def __init__(self, mean=0.0, std=1.0):
        self.mean = mean
        self.std = std

    def __call__(self, tensor):
        noise = torch.randn(tensor.size()) * self.std + self.mean
        return tensor + noise

    def __repr__(self):
        return f"{self.__class__.__name__}(mean={self.mean}, std={self.std})"

In [18]:
def visualize_generator(img_file, transform):
    # Load the requested image
    img = Image.open(img_file)

    # Apply transformations to the image
    transformed_img = transform(img)  # Apply the transform
    
    # Convert the transformed tensor back to PIL image
    transformed_img_pil = transforms.ToPILImage()(transformed_img)

    # Display the augmented image for verification
    grid = make_grid(transformed_img.unsqueeze(0), nrow=1)
    grid_np = grid.numpy().transpose(1, 2, 0)
    grid_np = (grid_np * 255).astype(np.uint8)

    # plt.figure(figsize=(4, 4))
    # plt.axis("off")
    # plt.imshow(grid_np)
    # plt.show()

    return transformed_img_pil

In [19]:
# Set the source and destination paths
train_path = "C:/Users/araya/Desktop/vscode/pretrained/dataset/train"
train_path = "C:/Users/araya/Desktop/preprocessing/train/frames"
test_path = "C:/Users/araya/Desktop/dunggy"

folder_list = os.listdir(train_path)

In [20]:
# Loop through each folder
for each_folder in folder_list:
    # Create the corresponding folder in the test path
    test_folder_path = os.path.join(test_path, each_folder)
    os.makedirs(test_folder_path, exist_ok=True)
    
    random_angle = random.uniform(-10, 10)
    print(f"Folder: {each_folder}, Angle: {random_angle}")
    
    file_path = os.path.join(train_path, each_folder)
    frame_list = os.listdir(file_path)

    # Define the transformation
    transform = transforms.Compose([
        transforms.ToTensor(),
        AddGaussianNoise(mean=0.0, std=0.1),  # Add Gaussian noise
        transforms.RandomRotation(degrees=(random_angle, random_angle)),
    ])

    # Loop through each frame in the folder
    for i, frame in enumerate(frame_list):
        each_frame_path = os.path.join(file_path, frame)
        print(f"Processing: {each_frame_path}")
        
        # Generate the augmented image
        for i in range(0,16,1):
            augmented_image = visualize_generator(each_frame_path + "/" + f"important_frame_{i}.jpg", transform)
        
            # Save the augmented image to the test folder
            save_path = os.path.join(test_folder_path, f"important_frame_{i}.jpg")
            augmented_image.save(save_path)
            print(f"Saved: {save_path}")

Folder: 1.mp4, Angle: 6.647345605887178
Processing: C:/Users/araya/Desktop/preprocessing/train/frames\1.mp4\video1
Saved: C:/Users/araya/Desktop/dunggy\1.mp4\important_frame_0.jpg
Saved: C:/Users/araya/Desktop/dunggy\1.mp4\important_frame_1.jpg
Saved: C:/Users/araya/Desktop/dunggy\1.mp4\important_frame_2.jpg
Saved: C:/Users/araya/Desktop/dunggy\1.mp4\important_frame_3.jpg
Saved: C:/Users/araya/Desktop/dunggy\1.mp4\important_frame_4.jpg
Saved: C:/Users/araya/Desktop/dunggy\1.mp4\important_frame_5.jpg
Saved: C:/Users/araya/Desktop/dunggy\1.mp4\important_frame_6.jpg
Saved: C:/Users/araya/Desktop/dunggy\1.mp4\important_frame_7.jpg
Saved: C:/Users/araya/Desktop/dunggy\1.mp4\important_frame_8.jpg
Saved: C:/Users/araya/Desktop/dunggy\1.mp4\important_frame_9.jpg
Saved: C:/Users/araya/Desktop/dunggy\1.mp4\important_frame_10.jpg
Saved: C:/Users/araya/Desktop/dunggy\1.mp4\important_frame_11.jpg
Saved: C:/Users/araya/Desktop/dunggy\1.mp4\important_frame_12.jpg
Saved: C:/Users/araya/Desktop/dunggy\

KeyboardInterrupt: 

### Test on Breaking Frame